In [1]:
from useful_functions import format_datetime64, get_daterange_str

import numpy as np
import pandas as pd
import xarray as xr
import os
from datetime import datetime

from metpy.calc import wind_direction, wind_speed
from metpy.units import units
from ICONProcessor import ICONDataGrid

data_out_dir = '../data_out/'

In [2]:
def get_var_attrs_ref(ref_ds_file, ref_var, res):
    # get reference dataset for meta data
    ref_ds = xr.open_dataset(ref_ds_file)
    attrs = {
        "long_name": ref_ds[ref_var].long_name,
        "standard_name": ref_ds[ref_var].standard_name,
        "units": ref_ds[ref_var].units,
        "temporal_resolution": f'{res} averages (origin is the last value of the timeseries)'
    }
    return attrs

## Script settings

In [3]:
# ICON
icon_nlev = 75 # lowest number of ICON model levels
icon_cid = 35704
icon_exp = 'v370_2030'
icon_file_extpar = '/Users/geoalxx/Python/_ICON_data/hef/grid/external_parameter_icon_hef_51m_DOM07_ALBEDO_RGI_2030_tiles.nc'
icon_file_30min = f'/Users/geoalxx/Python/glacier_space/data_icon_h3/levante_v370_2030/{icon_exp}_LES_30min_51m_ml_{icon_cid}.nc'  # data
icon_file = f'/Users/geoalxx/Python/glacier_space/data_icon_h3/levante_v370_2030/{icon_exp}_LES_51m_ml_{icon_cid}.nc'  # z_mc, z_ifc

# WindRanger
wr_dir = '/Users/geoalxx/HEFEX III/WindRanger/averaged/'
cnt_height = 13

## Load and prepare WindRanger data

In [4]:
wr_variables_ts = ['alt','lat','lon', 'heading', 'pitch', 'roll']
wr_variables_2D = ['U','V','W','DIR','VEL','SU','SV','SW','SNR','DQ']

In [5]:
wr_files = sorted(os.listdir(wr_dir))

# prepare data containers for time and time series variables
dim_h_wr = None
dim_time_wr = []
wr_df_ts = pd.DataFrame()

# prepare empty df for each 2D variable
wr_df_2D = {}
for v in wr_variables_2D:
    wr_df_2D[v] = pd.DataFrame()

for file in wr_files:
    if file.endswith('_averaged.nc'):
        # get time and height values
        ds = xr.open_dataset(wr_dir + file)
        time = pd.to_datetime(ds['time'].values, utc=True)
        height = ds['height'].values
        print('Processing', file,'- Height:', len(height), '- Time:', len(time))

        # do some quality checks
        if len(height) != cnt_height:
            print(f'Height dimension of {file} does not meet requirement. -> skipped!')
            continue
        elif dim_h_wr is None:
            dim_h_wr = height
            wr_file_ref = wr_dir + file
        elif not np.array_equal(height, dim_h_wr):
            print(f'Height dimension of {file} do not match. -> skipped!')
            continue

        # collect time series data
        dim_time_wr.extend(time)
        df_ts_tmp = pd.DataFrame(index=time)
        for v in wr_variables_ts:
            df_ts_tmp[v] = ds[v].values
        wr_df_ts = pd.concat([wr_df_ts, df_ts_tmp])

        # collect 2D data
        for v in wr_variables_2D:
            wr_df_2D[v] = pd.concat([wr_df_2D[v], pd.DataFrame(ds[v].values, index=time, columns=dim_h_wr)])

Processing 20250806_000000_averaged.nc - Height: 7 - Time: 23
Height dimension of 20250806_000000_averaged.nc does not meet requirement. -> skipped!
Processing 20250806_150000_averaged.nc - Height: 13 - Time: 53
Processing 20250807_000000_averaged.nc - Height: 13 - Time: 78
Processing 20250807_125000_averaged.nc - Height: 13 - Time: 66
Processing 20250808_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250809_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250810_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250811_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250812_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250813_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250814_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250815_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250816_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250817_000000_averaged.nc - Height: 13 - Time: 144
Processing 20250818_00

## Load and prepare ICON data

In [6]:
ds_30min = xr.open_dataset(icon_file_30min)
ds = xr.open_dataset(icon_file)
ds_extpar = xr.open_dataset(icon_file_extpar)
ds_cell = ds_extpar.isel(cell=icon_cid-1)

# dimension: time, height (full level and half level)
dim_time_i = pd.Series([ICONDataGrid.convert_float_dt(f) for f in ds_30min.time.values])
elev = ds.z_ifc.values[-1, 0]
dim_h_mc = ds.z_mc.values[-icon_nlev:, 0] - elev
dim_h_ifc = ds.z_ifc.values[-icon_nlev-1:, 0] - elev
print(f'Vertical extent ({icon_nlev} levels):',dim_h_ifc[-1],'-',dim_h_ifc[0],'m')

# get data
icon_df_u = pd.DataFrame(ds_30min.u.values[:, -icon_nlev:, 0], index=dim_time_i, columns=dim_h_mc)
icon_df_v = pd.DataFrame(ds_30min.v.values[:, -icon_nlev:, 0], index=dim_time_i, columns=dim_h_mc)
icon_df_w = pd.DataFrame(ds_30min.v.values[:, -icon_nlev - 1:, 0], index=dim_time_i, columns=dim_h_ifc)

# calc wind direction and speed
icon_df_wspd = pd.DataFrame(wind_speed(icon_df_u.values * units('m/s'), icon_df_v.values * units('m/s')), index=dim_time_i, columns=dim_h_mc)
icon_df_wdir = pd.DataFrame(wind_direction(icon_df_u.values * units('m/s'), icon_df_v.values * units('m/s')), index=dim_time_i, columns=dim_h_mc)

Vertical extent (75 levels): 0.0 - 2611.4626 m


 ## Build datasets

In [7]:
def create_data_array_2D(df, name, time_dim, height_dim, height_name, var_attrs):
    # get time from dataset
    time_naive = [dt.replace(tzinfo=None) for dt in df.index.to_numpy()]

    # create DataArray
    xr_data = xr.DataArray(
        df.values,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
            height_dim: (height_dim, df.columns.to_numpy(), {"long_name": height_name}),
        },
        dims=[time_dim, height_dim],
        name=name,
        attrs=var_attrs
    )
    return xr_data

def create_data_array_ts(s, name, time_dim, var_attrs):
    # get time from dataset
    time_naive = [dt.replace(tzinfo=None) for dt in s.index.to_numpy()]

    # create DataArray
    xr_data = xr.DataArray(
        s.values,
        coords={
            time_dim: (time_dim, time_naive, {"long_name": "Timestamp in UTC"}),
        },
        dims=[time_dim],
        name=name,
        attrs=var_attrs
    )
    return xr_data

### ICON

In [8]:
# Combine DataArrays
res = '30min'
ds_out = xr.Dataset({
    'u'    : create_data_array_2D(icon_df_u, 'u', 'time', 'height_mc', 'height at full level center', get_var_attrs_ref(icon_file, 'u', res)),
    'v'    : create_data_array_2D(icon_df_v, 'v', 'time', 'height_mc', 'height at full level center', get_var_attrs_ref(icon_file, 'v', res)),
    'w'    : create_data_array_2D(icon_df_w, 'w', 'time', 'height_ifc', 'height at half level center', get_var_attrs_ref(icon_file, 'w', res)),
    'wspd' : create_data_array_2D(icon_df_wspd, 'wspd', 'time', 'height_mc', 'height at full level center', get_var_attrs_ref(wr_file_ref, 'VEL', res)),
    'wdir' : create_data_array_2D(icon_df_wdir, 'wdir', 'time', 'height_mc', 'height at full level center', get_var_attrs_ref(wr_file_ref, 'DIR', res)),
})

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq="30min")
ds_out = ds_out.reindex(time=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Vertical wind profile from ICON-LES for HEFEX III campaign at Lidar location",
    "institution": "Humboldt-Universität zu Berlin",
    "source": "icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "experiment_id": f"hefex3/{icon_exp}/exp_R3B15_51m",
    "grid_cell": f"cell {icon_cid}/{ds_extpar.sizes['cell']} of domain hef_51m_DOM07.nc (lon, lat, elev): {ds_cell.lon.values:.6f}°E, {ds_cell.lat.values:.6f}°N, {ds_cell.topography_c.values:.2f}m",
    "StartTime": format_datetime64( ds_out.time.values[0] ),
    "StopTime":  format_datetime64( ds_out.time.values[-1] ),
    "comment": f"ICON simulation based on glacier outlines from OGGM SSP370, year 2030"
}

# Write
output_file = data_out_dir + f'H3_ICON_{icon_exp}_cid{icon_cid}_vertical_profile.nc'
output_file = data_out_dir + f'HEFEX3__ICON_{icon_exp}_cid{icon_cid}_{icon_nlev}ml__avg{res}_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/HEFEX3__ICON_v370_2030_cid35704_75ml__avg30min_20250805-20250905.nc.


In [9]:
ds_out

<xarray.Dataset> Size: 2MB
Dimensions:     (time: 1525, height_mc: 75, height_ifc: 76)
Coordinates:
  * time        (time) datetime64[ns] 12kB 2025-08-05 ... 2025-09-05T18:00:00
  * height_mc   (height_mc) float32 300B 2.575e+03 2.503e+03 ... 15.0 5.0
  * height_ifc  (height_ifc) float32 304B 2.611e+03 2.539e+03 ... 10.0 0.0
Data variables:
    u           (time, height_mc) float32 458kB 3.329 3.417 ... 0.6214 0.3577
    v           (time, height_mc) float32 458kB -19.19 -19.08 ... -3.817 -3.003
    w           (time, height_ifc) float32 464kB -19.29 -19.19 ... -3.817 -3.003
    wspd        (time, height_mc) float32 458kB 19.48 19.38 ... 3.867 3.024
    wdir        (time, height_mc) float32 458kB 350.2 349.8 ... 350.8 353.2
Attributes:
    title:          Vertical wind profile from ICON-LES for HEFEX III campaig...
    institution:    Humboldt-Universität zu Berlin
    source:         icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)
    history:        Created 2026-04-15
    contact:        alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-946...
    campaign:       HEFEX III
    experiment_id:  hefex3/v370_2030/exp_R3B15_51m
    grid_cell:      cell 35704/88640 of domain hef_51m_DOM07.nc (lon, lat, el...
    StartTime:      2025-08-05 00:00:00
    StopTime:       2025-09-05 18:00:00
    comment:        ICON simulation based on glacier outlines from OGGM SSP37...

### Observations from WindRanger

In [10]:
# empty dataset
res = '10min'
ds_out = xr.Dataset()

# add WindRanger data 2D
for v in wr_variables_2D:
    if v == 'DIR':
        var_name = 'wdir'
    elif v == 'VEL':
        var_name = 'wspd'
    else:
        var_name = str.lower(v)
    ds_out[var_name] = create_data_array_2D(wr_df_2D[v], var_name, 'time', 'height', 'WindRanger measurement height', get_var_attrs_ref(wr_file_ref, v, res))

# add WindRanger time series data
for v in ['heading','pitch','roll']:
    var_name = str.lower(v)
    ds_out[var_name] = create_data_array_ts(wr_df_ts[v], var_name, 'time', get_var_attrs_ref(wr_file_ref, v, res))

# clean time series (fill gaps and remove duplicates)
ds_out = ds_out.sortby("time")
ds_out = ds_out.sel(time=~ds_out.indexes["time"].duplicated())
time_full = pd.date_range(ds_out.time.min().values, ds_out.time.max().values, freq="10min")
ds_out = ds_out.reindex(time=time_full)

# Add global attributes
ds_out.attrs = {
    "title": f"Vertical wind profiles from WindRanger lidar acquired during HEFEX III campaign",
    "institution": "Humboldt-Universität zu Berlin",
    "source": f"METEK LiDAR Averaged Profile Data ({res}-averaged output from device)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "location": f"WindRanger (lon, lat, elev): {wr_df_ts['lon'].mean():.6f}°E, {wr_df_ts['lat'].mean():.6f}°N, {wr_df_ts['alt'].mean():.2f}m",
    "StartTime": format_datetime64( ds_out.time.values[0] ),
    "StopTime":  format_datetime64( ds_out.time.values[-1] ),
    "comment": f"Raw WindRanger output for 10min averages (no filtering or correction); Instantaneous values available on request"
}

# Write
output_file = data_out_dir + f'HEFEX3__Obs_WindRanger_L0__avg{res}_{get_daterange_str(ds_out)}.nc'
ds_out.to_netcdf(output_file)
print(f'Output file saved as {output_file}.')

Output file saved as ../data_out/HEFEX3__Obs_WindRanger_L0__avg10min_20250806-20250831.nc.


In [11]:
ds_out

<xarray.Dataset> Size: 2MB
Dimensions:  (time: 3562, height: 13)
Coordinates:
  * time     (time) datetime64[ns] 28kB 2025-08-06T15:10:00 ... 2025-08-31T08...
  * height   (height) float32 52B 7.0 10.0 15.0 20.0 ... 125.0 150.0 175.0 200.0
Data variables: (12/13)
    u        (time, height) float32 185kB 1.548 1.649 1.653 ... nan nan nan
    v        (time, height) float32 185kB 2.746 2.424 2.627 ... nan nan nan
    w        (time, height) float32 185kB -0.2633 -0.3431 -0.377 ... nan nan nan
    wdir     (time, height) float64 370kB 209.4 214.2 212.2 ... nan nan nan
    wspd     (time, height) float32 185kB 3.152 2.932 3.104 2.85 ... nan nan nan
    su       (time, height) float32 185kB 0.4945 0.5356 0.4754 ... nan nan nan
    ...       ...
    sw       (time, height) float32 185kB 0.2735 0.3484 0.3834 ... nan nan nan
    snr      (time, height) float64 370kB -20.41 -20.07 -19.9 ... -35.76 -33.81
    dq       (time, height) float64 370kB 1.0 1.0 1.0 1.0 ... 0.0 0.0 0.0 0.0
    heading  (time) float32 14kB 0.0 0.0 0.0 0.0 0.0 0.0 ... nan 0.0 0.0 nan 0.0
    pitch    (time) float32 14kB 0.1943 0.2033 0.189 ... 0.1875 nan 0.1194
    roll     (time) float32 14kB -0.6666 -0.672 -0.692 ... -2.131 nan -1.91
Attributes:
    title:        Vertical wind profiles from WindRanger lidar acquired durin...
    institution:  Humboldt-Universität zu Berlin
    source:       METEK LiDAR Averaged Profile Data (10min-averaged output fr...
    history:      Created 2026-04-15
    contact:      alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-...
    campaign:     HEFEX III
    location:     WindRanger (lon, lat, elev): 10.771662°E, 46.801624°N, 2720...
    StartTime:    2025-08-06 15:10:00
    StopTime:     2025-08-31 08:40:00
    comment:      Raw WindRanger output for 10min averages (no filtering or c...